# TTM-Based Accelerometry Classification System
## Production-Grade Foundation Model for UK Biobank Analysis

This notebook implements a complete end-to-end system for training IBM's Tiny Time Mixer (TTM) on UK Biobank accelerometry data.

**Key Features:**
- TTM-R2 with only 1M parameters (200x smaller than alternatives)
- 3-stage training: Linear Probe → LoRA → Full Fine-tuning
- Multi-task SSL: Arrow of Time, Permutation, Time Warping
- Optimized for Google Colab (12GB RAM, T4 GPU)
- Target: >0.85 F1 on CAPTURE-24 benchmark

**Performance Targets:**
- Inference: <10ms per window on T4 GPU
- Memory: <3GB for batch=64
- Model size: <10MB serialized

## 1. Environment Setup

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Not running in Colab")

In [ ]:
# Clone repository (if in Colab)
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/AccelomtryFoundationModel.git
    %cd AccelomtryFoundationModel

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

print("✓ Dependencies installed")

In [ ]:
# Mount Google Drive for checkpointing
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ttm_accelerometry'
    !mkdir -p {DRIVE_DIR}/checkpoints
    !mkdir -p {DRIVE_DIR}/data
    print(f"✓ Mounted Drive: {DRIVE_DIR}")
else:
    DRIVE_DIR = './drive'
    !mkdir -p {DRIVE_DIR}/checkpoints
    !mkdir -p {DRIVE_DIR}/data

In [ ]:
# Imports
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import our modules
from src.data.data_loader import AccelerometryDataLoader, StreamingDataset, create_stratified_splits
from src.data.augmentations import SSLAugmentations, TTMDataAugmentor
from src.models.ttm_classifier import TTMAccelerometryClassifier, FocalLoss, create_class_weights
from src.training.trainer import ThreeStageTrainer
from src.training.colab_trainer import ColabOptimizedTrainer, setup_colab_environment
from src.evaluation.evaluator import AccelerometryEvaluator, benchmark_capture24
from src.utils.config import Config

# Set random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Load or create configuration
config = Config()

# Override for Colab
config.training.batch_size = 64
config.training.use_amp = True
config.colab.use_gradient_checkpointing = True
config.colab.target_gpu_memory_gb = 3.0

# Save config
config.to_yaml('config.yaml')

print(config)

## 3. Data Preparation

In [ ]:
# Initialize data loader
data_loader = AccelerometryDataLoader(
    data_dir=config.data.data_dir,
    cache_dir=config.data.cache_dir,
    window_size=config.data.window_size,
    stride=config.data.stride,
    sample_rate=config.data.sample_rate,
    use_cache=config.data.use_cache,
)

print(f"✓ Data loader initialized")
print(f"  Window size: {config.data.window_size} samples ({config.data.window_size/config.data.sample_rate:.3f}s)")
print(f"  Stride: {config.data.stride} samples (50% overlap)")

In [ ]:
# For demo: Create synthetic dataset
# In production, replace with actual UK Biobank .cwa files

print("Creating synthetic dataset for demonstration...")

import h5py
n_samples = 10000
window_size = 820
n_channels = 3

# Create synthetic data
windows = np.random.randn(n_samples, window_size, n_channels).astype(np.float32)
labels = np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.3, 0.4, 0.2, 0.1])

# Save to HDF5
os.makedirs('data', exist_ok=True)
with h5py.File('data/synthetic_data.h5', 'w') as f:
    f.create_dataset('windows', data=windows, compression='gzip')
    f.create_dataset('labels', data=labels)
    f.attrs['window_size'] = window_size
    f.attrs['sample_rate'] = 100

print(f"✓ Created synthetic dataset: {n_samples} samples")
print(f"  Class distribution: {np.bincount(labels)}")

In [ ]:
# Create train/val/test splits
train_idx, val_idx, test_idx = create_stratified_splits(
    'data/synthetic_data.h5',
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=SEED,
)

print(f"✓ Created splits:")
print(f"  Train: {len(train_idx)} samples")
print(f"  Val:   {len(val_idx)} samples")
print(f"  Test:  {len(test_idx)} samples")

In [ ]:
# Create datasets
train_dataset = StreamingDataset('data/synthetic_data.h5', indices=train_idx)
val_dataset = StreamingDataset('data/synthetic_data.h5', indices=val_idx)
test_dataset = StreamingDataset('data/synthetic_data.h5', indices=test_idx)

print(f"✓ Created datasets")

# Test loading
x, y = train_dataset[0]
print(f"  Sample shape: {x.shape}, label: {y}")

## 4. Model Initialization

In [ ]:
# Create model
model = TTMAccelerometryClassifier(
    model_name=config.model.model_name,
    n_classes=config.model.n_classes,
    n_channels=config.model.n_channels,
    context_length=config.model.context_length,
    hidden_dim=config.model.hidden_dim,
    dropout=config.model.dropout,
    freeze_encoder=config.model.freeze_encoder,
)

model = model.to(device)

# Print model statistics
stats = model.get_parameter_stats()
print(f"\n✓ Model initialized:")
print(f"  Total parameters:     {stats['total']:,}")
print(f"  Trainable parameters: {stats['trainable']:,}")
print(f"  Encoder parameters:   {stats['encoder_total']:,}")
print(f"  Classifier parameters: {stats['classifier']:,}")
print(f"  Model size: ~{stats['total'] * 4 / 1e6:.1f} MB (FP32)")

In [ ]:
# Test forward pass
print("Testing forward pass...")

test_batch = torch.randn(8, 820, 3).to(device)
with torch.cuda.amp.autocast():
    output = model(test_batch, return_embeddings=True)

print(f"✓ Forward pass successful:")
print(f"  Input shape:      {test_batch.shape}")
print(f"  Logits shape:     {output['logits'].shape}")
print(f"  Embeddings shape: {output['embeddings'].shape}")

# Memory usage
if torch.cuda.is_available():
    memory_gb = torch.cuda.memory_allocated() / 1e9
    print(f"  GPU memory: {memory_gb:.2f} GB")

## 5. Loss Function with Class Weights

In [ ]:
# Calculate class weights from training data
train_labels_all = labels[train_idx]
class_counts = {i: (train_labels_all == i).sum() for i in range(4)}

print(f"Class distribution:")
for i, count in class_counts.items():
    print(f"  Class {i}: {count} samples ({count/len(train_labels_all)*100:.1f}%)")

# Create class weights
class_weights = create_class_weights(class_counts, mode='inverse')
class_weights = class_weights.to(device)

print(f"\nClass weights: {class_weights.cpu().numpy()}")

# Create focal loss
criterion = FocalLoss(alpha=class_weights, gamma=2.0)

print(f"✓ Focal loss initialized (gamma=2.0)")

## 6. Three-Stage Training Pipeline

In [ ]:
# Initialize Colab-optimized trainer
from torch.utils.data import DataLoader

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Initialize trainer
trainer = ThreeStageTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    checkpoint_dir=f"{DRIVE_DIR}/checkpoints",
    use_amp=config.training.use_amp,
    gradient_accumulation_steps=config.training.gradient_accumulation_steps,
    max_grad_norm=config.training.max_grad_norm,
)

print(f"✓ Trainer initialized")

In [ ]:
# Run 3-stage training pipeline
# NOTE: Reduce epochs for quick demo. Use full epochs for production.

history = trainer.run_full_pipeline(
    stage1_epochs=2,   # Production: 10
    stage2_epochs=2,   # Production: 20
    stage3_epochs=2,   # Production: 10
    stage1_lr=1e-3,
    stage2_lr=1e-4,
    stage3_lr=1e-5,
)

## 7. Training Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/training_curves.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved training curves to {DRIVE_DIR}/training_curves.png")

## 8. Evaluation on Test Set

In [ ]:
# Create test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Initialize evaluator
evaluator = AccelerometryEvaluator(
    class_names=['Sleep', 'Sedentary', 'Light', 'MVPA'],
    n_bootstrap=1000,
    confidence_level=0.95,
)

# Evaluate
metrics = evaluator.evaluate(
    model=model,
    dataloader=test_loader,
    device=device,
    return_predictions=True,
    use_amp=True,
)

In [ ]:
# Print results
evaluator.print_results(metrics)

In [ ]:
# Plot confusion matrix
evaluator.plot_confusion_matrix(
    metrics,
    save_path=f"{DRIVE_DIR}/confusion_matrix.png",
    normalize=True,
)

## 9. CAPTURE-24 Benchmark

In [ ]:
# Benchmark against CAPTURE-24 standard
benchmark_results = benchmark_capture24(
    model=model,
    test_loader=test_loader,
    device=device,
    target_f1=0.85,  # Target: >0.85 F1
)

## 10. Inference Speed Benchmark

In [ ]:
# Benchmark inference speed
import time

model.eval()
test_input = torch.randn(1, 820, 3).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(test_input)

# Benchmark
n_runs = 100
times = []

for _ in range(n_runs):
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            _ = model(test_input)
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    times.append(time.time() - start)

mean_time = np.mean(times) * 1000  # ms
std_time = np.std(times) * 1000

print(f"\n{'='*50}")
print(f"INFERENCE SPEED BENCHMARK")
print(f"{'='*50}")
print(f"Per window (batch=1):")
print(f"  Mean: {mean_time:.2f} ± {std_time:.2f} ms")
print(f"  Target: <10ms on T4 GPU")
print(f"  Status: {'✓ PASS' if mean_time < 10 else '✗ FAIL'}")
print(f"{'='*50}")

## 11. Save Final Model

In [ ]:
# Save model
model_path = f"{DRIVE_DIR}/ttm_accelerometry_final.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'metrics': metrics,
    'class_names': ['Sleep', 'Sedentary', 'Light', 'MVPA'],
}, model_path)

model_size_mb = os.path.getsize(model_path) / 1e6

print(f"✓ Saved final model to {model_path}")
print(f"  Size: {model_size_mb:.2f} MB")
print(f"  Target: <10MB")
print(f"  Status: {'✓ PASS' if model_size_mb < 10 else '✗ FAIL'}")

## 12. Summary Report

In [ ]:
# Generate summary report
print(f"\n{'='*70}")
print(f"TTM ACCELEROMETRY SYSTEM - FINAL REPORT")
print(f"{'='*70}")

print(f"\n1. MODEL ARCHITECTURE:")
print(f"   Base model:           {config.model.model_name}")
print(f"   Total parameters:     {stats['total']:,}")
print(f"   Trainable parameters: {stats['trainable']:,}")
print(f"   Model size:           {model_size_mb:.2f} MB")

print(f"\n2. TRAINING:")
print(f"   Training samples:     {len(train_idx):,}")
print(f"   Validation samples:   {len(val_idx):,}")
print(f"   Test samples:         {len(test_idx):,}")
print(f"   Batch size:           {config.training.batch_size}")
print(f"   Mixed precision:      {config.training.use_amp}")

print(f"\n3. PERFORMANCE:")
print(f"   Test F1 (macro):      {metrics['f1_macro']:.4f} [{metrics['f1_macro_ci'][0]:.4f}, {metrics['f1_macro_ci'][1]:.4f}]")
print(f"   Test Accuracy:        {metrics['accuracy']:.4f}")
print(f"   Inference time:       {mean_time:.2f} ± {std_time:.2f} ms/window")
print(f"   GPU memory (batch=64): ~{memory_gb:.2f} GB")

print(f"\n4. BENCHMARKS:")
print(f"   CAPTURE-24 target:    0.85 F1")
print(f"   Achieved:             {metrics['f1_macro']:.4f}")
print(f"   Status:               {'✓ PASS' if metrics['f1_macro'] >= 0.85 else '⚠ NEEDS IMPROVEMENT'}")

print(f"\n5. PER-CLASS RESULTS:")
for class_name, class_metrics in metrics['per_class'].items():
    print(f"   {class_name:12s}: F1={class_metrics['f1']:.4f}, "
          f"Prec={class_metrics['precision']:.4f}, "
          f"Rec={class_metrics['recall']:.4f}")

print(f"\n{'='*70}")
print(f"\n✓ Training complete! Model saved to {model_path}")
print(f"{'='*70}\n")